## 1. Loading the Dataset
- Run the RunBugRun from Hugging Face.
- Convert the original data set into only the columns that we need.
- Kept certain columns like buggy/fixed code for error analysis.


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path("../").resolve()))
from src.preprocessing import generate_diff
from datasets import load_dataset
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("iberu/RunBugRun")

buggy_code = []
fixed_code = []
diff_generated = []
labels_generated = []
full_label = []

excluded_v1_classes = ['literal', 'function', 'variable_access', 'io', 'try_catch']

for row in ds['train']:
    if not row['labels']:
        continue

    if len(row['labels']) != 1:
        continue

    top_level_label = row['labels'][0].split('.')[0]

    if top_level_label in excluded_v1_classes:
        continue
    buggy_code.append(row['buggy_code'])
    fixed_code.append(row['fixed_code'])
    diff_generated.append(generate_diff(row['buggy_code'], row['fixed_code']))
    labels_generated.append(top_level_label)
    full_label.append(row['labels'][0])

cleaned_df = pd.DataFrame({
    'buggy_code': buggy_code,
    'fixed_code': fixed_code,
    'diff': diff_generated,
    'top_level_label': labels_generated,
    'full_label': full_label
})

cleaned_df.to_csv("../data/processed/cleaned_v1.csv", index=False)


## Checking the Dataset (HuggingFace)
- Verify that all of the information is correct such as: 
- amount of examples and targets matching
- checking how many examples are left over after filtering

In [3]:
from collections import Counter
print(f'Amount of diff_generated: {len(diff_generated)}')
print(f'Amount of labels_generated: {len(labels_generated)}')
print('Printing out the first 5 examples from diff:')
for i in range(5):
    print(diff_generated[i])

print('Printing out the first 5 labels:')
for i in range(5):
    print(labels_generated[i])
    
print(f'Total generated diffs: {len(diff_generated)}')
print(f"Total training examples: {len(ds['train'])}")
print(f'Usable examples: {len(diff_generated)}')
print(f"Filtered out: {len(ds['train']) - len(diff_generated)}")

label_counts = Counter(labels_generated)
print(label_counts)

Amount of diff_generated: 35641
Amount of labels_generated: 35641
Printing out the first 5 examples from diff:
-         print(i,"x",j,"=",i*j)
+         print(i,"x",j,"=",i*j,sep="")
- [print('{}x{}={}').format(i, j, i * j) for j in range(1, 10) for i in range(1, 10)]
+ [print('{}x{}={}'.format(i, j, i * j)) for i in range(1, 10) for j in range(1, 10)]
- for i in range(10):
+ for i in range(1, 10):
-     for j in range(10):
+     for j in range(1, 10):
- 
- 		print(i,"x",j,"=",i*j,)
+ 		print(i,"x",j,"=",i*j,sep="")
+ #coding: UTF-8
+ 
- for i in range(10):
+ for i in range(1, 10):
- 	for j in range(10):
+ 	for j in range(1, 10):
Printing out the first 5 labels:
call
call
call
call
call
Total generated diffs: 35641
Total training examples: 133705
Usable examples: 35641
Filtered out: 98064
Counter({'call': 17165, 'expression': 7415, 'control_flow': 5394, 'assignment': 4586, 'identifier': 1081})
